# JobShield – 04 Red-flag Features and XGBoost (Week 4)
**Group 5 · AML-3403 AI and ML Capstone · Week 4 lead: Narinder Kaur** · https://github.com/Krushal16/JobShield

Research question 2: *do engineered red-flag features improve detection over text alone?*

1. Red-flag features – which warning signs really go with fraud? *(Arti)*
2. Train and compare: Week 3 baseline vs XGBoost (text only) vs XGBoost (text + red flags) *(Diksha)*
3. Precision-recall curves and feature importance *(Diksha)*
4. Which missed scams do red flags fix? *(Narinder)*
5. Save the model and explain single postings with red flags *(Krushal)*
6. Summary for Weekly Report 4

Needs `data/processed/train.csv` and `val.csv` (notebook 02) and `xgboost` (pre-installed on Colab). Test set untouched.

In [2]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data.loader import TARGET
from src.data.split import load_splits
from src.features.red_flags import red_flag_fraud_rates
from src.training.train_xgb import train_and_compare, BASELINE, XGB_TEXT, XGB_FLAGS
from src.training.train_baseline import save_model
from src.models.xgb import feature_importance
from src.evaluation.metrics import plot_confusion, plot_pr_curve
from src.evaluation.error_analysis import compare_misses, missed_scam_profile
from src.inference.predict import predict_posting, load_model
from src.utils.config import load_config, project_path, set_seed
CFG = load_config(); set_seed(CFG["seed"])
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
FIG = project_path(CFG["paths"]["figures_dir"]); FIG.mkdir(parents=True, exist_ok=True)
train_df, val_df, _ = load_splits()
y_val = val_df[TARGET].values
summary = {}

ModuleNotFoundError: No module named 'src.evaluation.error_analysis'

## 1. Red-flag features – fraud rate with vs without each flag (training set only)

In [ ]:
rates = red_flag_fraud_rates(train_df)
plot_rates = rates[rates.enough_data].head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.8))
yy = np.arange(len(plot_rates))
ax.barh(yy + 0.2, plot_rates["fraud_rate_with_%"], height=0.4, color="#C55A11", label="flag present")
ax.barh(yy - 0.2, plot_rates["fraud_rate_without_%"], height=0.4, color="#9DC3E6", label="flag absent")
ax.set_yticks(yy); ax.set_yticklabels(plot_rates.label)
ax.set_xlabel("Fraud rate (%)"); ax.set_title("Fraud rate with vs without each red flag (train)"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(FIG / "12_red_flag_fraud_rates.png", dpi=200); plt.show()
summary["top_red_flags"] = [f"{r.label}: {r['fraud_rate_with_%']}% vs {r['fraud_rate_without_%']}% (x{r.lift})"
                            for _, r in rates[rates.enough_data].head(6).iterrows()]
rates

## 2. Train and compare the three models (a few minutes)

In [ ]:
out = train_and_compare(train_df, val_df, CFG, log=True)
res = pd.DataFrame(out["results"])[["model", "notes", "precision", "recall", "f1", "pr_auc", "tp", "fn", "fp"]]
res.round(3)

## 3. Precision-recall curves, confusion matrix and feature importance

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for name in (BASELINE, XGB_TEXT):
    plot_pr_curve(y_val, out["probs"][name], label=name.replace(" (Week 3 baseline)", ""), ax=ax)
plot_pr_curve(y_val, out["probs"][XGB_FLAGS], label=XGB_FLAGS, ax=ax, save_as="13_xgb_pr_curves.png"); plt.show()
best = [r for r in out["results"] if r["model"] == XGB_FLAGS and "F1-best" in r["notes"]][0]
plot_confusion(best, save_as="14_xgb_confusion.png"); plt.show()
imp = feature_importance(out["models"][XGB_FLAGS], n=20)
fig, ax = plt.subplots(figsize=(6.5, 5))
colors = ["#C55A11" if s == "red flag" else "#2E74B5" for s in imp.source[::-1]]
ax.barh(imp.feature[::-1], imp.importance[::-1], color=colors)
ax.set_title("XGBoost – top 20 features (orange = red flag, blue = word)"); ax.set_xlabel("importance")
plt.tight_layout(); plt.savefig(FIG / "15_xgb_feature_importance.png", dpi=200); plt.show()
summary["red_flags_in_top20"] = int((imp.source == "red flag").sum())
imp

## 4. Which missed scams do red flags fix?

In [ ]:
th = out["thresholds"]
cmp_base = compare_misses(y_val, out["probs"][BASELINE], th[BASELINE], out["probs"][XGB_FLAGS], th[XGB_FLAGS])
cmp_text = compare_misses(y_val, out["probs"][XGB_TEXT], th[XGB_TEXT], out["probs"][XGB_FLAGS], th[XGB_FLAGS])
summary["baseline_vs_xgb_flags"] = cmp_base
summary["xgb_text_vs_xgb_flags"] = cmp_text
print("Baseline (A) vs XGBoost + red flags (B):", cmp_base)
print("XGBoost text only (A) vs XGBoost + red flags (B):", cmp_text)
missed_scam_profile(val_df, out["probs"][XGB_FLAGS], threshold=th[XGB_FLAGS])

## 5. Save the model and explain single postings

In [ ]:
path = save_model(out["models"][XGB_FLAGS], th[XGB_FLAGS], cfg=CFG, kind="xgb", extra={"xgboost": CFG["xgboost"]})
bundle = load_model(path, model="xgb")
demo = predict_posting(title="Data Entry Assistant - Work From Home",
                       description="Earn $500 weekly from home. No experience needed. Contact us on WhatsApp to start "
                                   "immediately. A cheque for equipment will be sent to you.",
                       has_company_logo=0, model_bundle=bundle)
summary["demo"] = {k: demo[k] for k in ("probability_fake", "label", "risk_level", "red_flags")}
demo

## 6. Summary for Weekly Report 4

In [ ]:
for r in out["results"]:
    summary[f"{r['model']} | {r['notes']}"] = f"P={r['precision']:.3f} R={r['recall']:.3f} F1={r['f1']:.3f} PR-AUC={r['pr_auc']:.3f} TP={r['tp']} FN={r['fn']} FP={r['fp']}"
for k, v in summary.items():
    print(f"{k}: {v}")